In [6]:
import os

# 1. Ensure the docs directory exists
os.makedirs("docs", exist_ok=True)

# 2. Define the exact file path
file_path = "docs/prompt_ladder.md"

# 3. Define the complete prompt engineering ladder content
content = """# Prompt Engineering Ladder: ML Data Leakage Audit

## 0. Weak Baseline Prompt

```text
Help me find data leakage in my machine learning code.
"""

In [11]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 1. Load dataset (replace with your actual file path/URL)
url = "https://huggingface.co/datasets/FlyRank/internship-starter/raw/main/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()

# 2. Define column names present in the dataset
page_col = 'page_id' if 'page_id' in df.columns else df.columns[0] # 'content_id'
pos_col = 'avg_position' if 'avg_position' in df.columns else 'position' # 'avg_position'

# Correcting click_col based on available columns
if 'clicks_90d' in df.columns:
    click_col = 'clicks_90d'
elif 'clicks' in df.columns: # fallback if 'clicks' was used in other datasets
    click_col = 'clicks'
else:
    # Fallback to df.columns[1] if no appropriate click column is found, but this might not be a meaningful feature.
    print("Warning: No 'clicks' or 'clicks_90d' column found. Using 'client_id' as a fallback, which might not be appropriate for modeling.")
    click_col = df.columns[1] # This would be 'client_id' for this dataset

# 3. Sort chronologically to preserve time-series ordering
# This step is likely not useful as 'week_ending' is not in columns and 'content_id' is unique.
# Keeping it for now but noting its limited effect in this specific dataset context.
if 'week_ending' in df.columns:
    df = df.sort_values([page_col, 'week_ending'])

# The previous feature engineering steps (position_shift, trend_direction, target_next_pos)
# relied on grouping by 'page_col' and performing time-series operations (diff, shift).
# However, the current dataset has only one entry per 'content_id' (which is 'page_col').
# This means that 'diff()' and 'shift(-1)' operations within each group result in
# all NaN values for 'position_shift' and 'target_next_pos'.
# Consequently, df.dropna() on these columns results in an empty DataFrame,
# leading to X and y being empty and the ValueError.

# To fix this, we will adapt the feature engineering for a cross-sectional dataset.
# We will use existing non-NaN columns as features and define a target based on current data.

# 4. Define feature matrix (X)
# We will use 'pos_col' (avg_position) and the corrected 'click_col' (clicks_90d) directly as features.
# 'position_shift' and 'trend_direction' are removed as they are all NaNs due to unique page_col.
features = [pos_col, click_col]
X = df[features].dropna() # Drop rows where essential features themselves might be NaN

# 5. Define target vector (y)
# Since 'target_next_pos' cannot be computed meaningfully, we define a target based on the current 'avg_position'.
# For example, classify if the 'avg_position' is good (e.g., less than or equal to 10) or bad (greater than 10).
# The threshold of 10 is inferred from the previous y = (df['position'].shift(-1) > 10).astype(int)
y = (df[pos_col] > 10).astype(int)

# Align y with X after dropping NaNs in X to ensure consistent indices
y = y.loc[X.index]

# 6. Ensure X and y are not empty before fitting the model
if X.empty or y.empty:
    raise ValueError("X or y is empty after preprocessing. This indicates an issue with data or feature selection. Please check the dataset or feature column names.")

# 7. Fit Model
model = RandomForestClassifier(random_state=42)
model.fit(X, y)

print("Model trained successfully!")

Model trained successfully!


In [14]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv(url)
df['position_shift'] = df.groupby(page_col)[pos_col].diff()
df['trend_direction'] = (df['position_shift'] < 0).astype(int)

X = df[[pos_col, click_col, 'position_shift', 'trend_direction']]
y = (df[pos_col].shift(-1) > 10).astype(int)

model = RandomForestClassifier()
model.fit(X, y)

RandomForestClassifier()